Chaque camion a une teneur réelle 𝑍, mais la décision est prise sur une teneur échantillonnée 𝑍∗
Si on envoie à l’usine les camions dont  𝑍∗ >𝑇𝑐, alors la moyenne des 𝑍∗ des camions retenus surestime la moyenne réelle des camions effectivement envoyés à l’usine.
C’est un biais conditionnel créé par la sélection sur une estimation imparfaite.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display


def demo_biais_conditionnel(
    mu=1.0,          # moyenne réelle des camions
    sigma=0.35,      # variabilité réelle entre camions
    bruit=0.25,      # bruit d'échantillonnage : Z* = Z + erreur
    tc=1.3,          # teneur de coupure
    n_camions=800,   # nombre de camions simulés
    seed=0
):
    rng = np.random.default_rng(seed)

    # ----------------------------
    # 1) Réalité : teneur vraie des camions
    # ----------------------------
    z_reel = rng.normal(mu, sigma, n_camions)

    # ----------------------------
    # 2) Echantillonnage / estimation
    # ----------------------------
    erreur = rng.normal(0, bruit, n_camions)
    z_est = z_reel + erreur

    # ----------------------------
    # 3) Décision sur l'estimation
    # ----------------------------
    usine = z_est > tc
    sterile = ~usine

    # Cas particuliers
    bons_usine = usine & (z_reel > tc)
    dilution = usine & (z_reel <= tc)
    pertes = sterile & (z_reel > tc)

    # ----------------------------
    # 4) Statistiques
    # ----------------------------
    if usine.sum() > 0:
        mean_est_usine = z_est[usine].mean()
        mean_real_usine = z_reel[usine].mean()
        biais = mean_est_usine - mean_real_usine
    else:
        mean_est_usine = np.nan
        mean_real_usine = np.nan
        biais = np.nan

    dilution_pct = 100 * dilution.mean()
    pertes_pct = 100 * pertes.mean()
    prop_usine = 100 * usine.mean()

    # ----------------------------
    # 5) Figure légère : 2 lignes x 2 colonnes
    # ----------------------------
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    fig.suptitle("Biais conditionnel sur sélection des camions", fontsize=15)

    # --- (1) Nuage Z* vs Z
    ax = axes[0, 0]
    ax.scatter(z_reel, z_est, s=12, alpha=0.35)
    lim_min = min(z_reel.min(), z_est.min(), tc) - 0.2
    lim_max = max(z_reel.max(), z_est.max(), tc) + 0.2
    ax.plot([lim_min, lim_max], [lim_min, lim_max], linestyle="--", linewidth=2, label="y = x")
    ax.axvline(tc, linewidth=2, label="Tc réel")
    ax.axhline(tc, linewidth=2, label="Tc estimé")
    ax.set_xlim(lim_min, lim_max)
    ax.set_ylim(lim_min, lim_max)
    ax.set_xlabel("Teneur réelle Z")
    ax.set_ylabel("Teneur estimée Z*")
    ax.set_title("Décision prise sur l'échantillonnage")
    ax.legend(fontsize=9)

    # --- (2) Histogrammes à l'usine
    ax = axes[0, 1]
    if usine.sum() > 0:
        bins = np.linspace(
            min(z_reel[usine].min(), z_est[usine].min()),
            max(z_reel[usine].max(), z_est[usine].max()),
            18
        )
        ax.hist(z_reel[usine], bins=bins, density=False, histtype="step", linewidth=2, label="Réel à l'usine")
        ax.hist(z_est[usine], bins=bins, density=False, histtype="step", linewidth=2, label="Échantillonné à l'usine")
        ax.axvline(mean_real_usine, linewidth=2, linestyle="--", label=f"Moy. réelle = {mean_real_usine:.2f}")
        ax.axvline(mean_est_usine, linewidth=2, linestyle="-.", label=f"Moy. échant. = {mean_est_usine:.2f}")
    ax.axvline(tc, linewidth=2, label="Tc")
    ax.set_title("Camions envoyés à l'usine")
    ax.set_xlabel("Teneur")
    ax.set_ylabel("Nombre de camions")
    ax.legend(fontsize=9)

    # --- (3) Barres de moyennes
    ax = axes[1, 0]
    labels = ["Moy. réelle\nà l'usine", "Moy. échant.\nà l'usine"]
    values = [mean_real_usine, mean_est_usine]
    ax.bar(labels, values)
    ax.axhline(tc, linewidth=2, linestyle="--", label="Tc")
    ax.set_title("Sur les camions sélectionnés usine")
    ax.set_ylabel("Teneur moyenne")
    ax.legend(fontsize=9)

    # --- (4) Texte de synthèse
    ax = axes[1, 1]
    ax.axis("off")
    txt = (
        f"Nombre de camions simulés : {n_camions}\n"
        f"Proportion envoyée à l'usine : {prop_usine:.1f} %\n\n"
        f"Moyenne réelle à l'usine        : {mean_real_usine:.3f}\n"
        f"Moyenne échantillonnée à l'usine: {mean_est_usine:.3f}\n"
        f"Biais conditionnel              : {biais:.3f}\n\n"
        f"Dilution   (Z* > Tc mais Z <= Tc) : {dilution_pct:.1f} %\n"
        f"Pertes     (Z* <= Tc mais Z > Tc) : {pertes_pct:.1f} %\n\n"
        f"Bruit = écart-type de l'erreur d'échantillonnage\n"
        f"avec Z* = Z + erreur"
    )
    ax.text(0.02, 0.98, txt, va="top", ha="left", fontsize=11, family="monospace")

    plt.tight_layout()
    plt.show()
    plt.close(fig)


# ----------------------------
# Widgets légers
# ----------------------------
w_mu = widgets.FloatSlider(
    value=1.0, min=0.6, max=1.4, step=0.05,
    description="Moyenne μ", continuous_update=False
)

w_sigma = widgets.FloatSlider(
    value=0.35, min=0.10, max=0.80, step=0.05,
    description="Variabilité", continuous_update=False
)

w_bruit = widgets.FloatSlider(
    value=0.25, min=0.00, max=0.60, step=0.02,
    description="Bruit", continuous_update=False
)

w_tc = widgets.FloatSlider(
    value=1.3, min=0.8, max=1.8, step=0.05,
    description="Tc", continuous_update=False
)

w_n = widgets.IntSlider(
    value=800, min=200, max=3000, step=100,
    description="Nb camions", continuous_update=False
)

w_seed = widgets.IntSlider(
    value=0, min=0, max=50, step=1,
    description="Seed", continuous_update=False
)

out = widgets.interactive_output(
    demo_biais_conditionnel,
    {
        "mu": w_mu,
        "sigma": w_sigma,
        "bruit": w_bruit,
        "tc": w_tc,
        "n_camions": w_n,
        "seed": w_seed
    }
)

ui = widgets.VBox([
    widgets.HTML("<h3>Biais conditionnel sur sélection des camions de 35 t</h3>"),
    widgets.HTML(
        "<b>Hypothèse :</b> on décide usine / stérile sur l'échantillonnage Z* ; "
        "on compare ensuite la moyenne échantillonnée et la moyenne réelle des camions envoyés à l'usine."
    ),
    widgets.HBox([w_mu, w_sigma, w_bruit]),
    widgets.HBox([w_tc, w_n, w_seed]),
    out
])

display(ui)